### Initialization

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
import time
from databricks.vector_search.client import VectorSearchClient

CATALOG = "humanoid_robotics"
SILVER  = f"{CATALOG}.silver"

SOURCE_TABLE    = f"{SILVER}.doc_chunks"
INDEX_NAME      = f"{SILVER}.doc_chunks_index"
ENDPOINT_NAME   = "humanoid_vs_endpoint"
EMBEDDING_MODEL = "databricks-gte-large-en"   # check the name under Serving

# Columns copied into the index. The list is fixed when the index is created:
# after changing it, set RECREATE_INDEX = True for one run, then back to False.
COLUMNS_TO_SYNC = [
    "chunk_id", "doc_id", "chunk_index", "chunk_text", "embedding_text",
    "title", "url", "publisher", "source_type", "language",
    "publish_date", "publish_year", "company_ids", "company_types", "companies_info",
]
RECREATE_INDEX = False

vsc = VectorSearchClient(disable_notice=True)

### Create Vector Search Endpoint

In [0]:

if not vsc.endpoint_exists(ENDPOINT_NAME):
    vsc.create_endpoint_and_wait(name=ENDPOINT_NAME, endpoint_type="STANDARD", verbose=True)

print(vsc.get_endpoint(ENDPOINT_NAME)["endpoint_status"])

### Create Delta Sync Index(embeddings happen here)

In [0]:
if RECREATE_INDEX and vsc.index_exists(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME):
    # Needed when COLUMNS_TO_SYNC (or the embedding setup) changes
    vsc.delete_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    while vsc.index_exists(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME):
        print("waiting for the old index to be deleted...")
        time.sleep(15)

if not vsc.index_exists(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME):
    # First run: creating the index also embeds every row of the source table
    vsc.create_delta_sync_index_and_wait(
        endpoint_name=ENDPOINT_NAME,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        primary_key="chunk_id",
        pipeline_type="TRIGGERED",
        embedding_source_column="embedding_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=COLUMNS_TO_SYNC,
        verbose=True,
    )
else:
    # Later runs: pick up changes in doc_chunks since the last sync (via Change Data Feed)
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    index.sync()
    index.wait_until_ready(verbose=True, wait_for_updates=True)

print(vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME).describe()["status"])

### Sync (after docs_chunks is rebuilt)

In [0]:
# Run this whenever 13_silver_doc_chunks has rewritten the table.
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
index.sync()
index.wait_until_ready(verbose=True, wait_for_updates=True)